Objetivo



Desarrolla una solución completa utilizando Proximal Policy Optimization (PPO) para optimizar el consumo energético de un centro de datos (Data Center).



El objetivo es reemplazar el enfoque basado en Deep Q-Learning (DQN) por un agente entrenado mediante Proximal Policy Optimization (PPO), uno de los algoritmos más robustos de Reinforcement Learning moderno.



El resultado debe ser un proyecto completamente funcional, modular y bien documentado.



Contexto del problema



El objetivo consiste en minimizar el consumo energético requerido para mantener la temperatura de un servidor dentro del rango óptimo de funcionamiento.



La temperatura interna del servidor depende de tres variables principales:



temperatura atmosférica

número de usuarios conectados

tasa de transferencia de datos



La temperatura puede aproximarse mediante una combinación lineal de estas tres variables.



El sistema tradicional de refrigeración consume energía proporcional al cambio absoluto de temperatura entre dos instantes.



El agente PPO deberá aprender una política que permita mantener la temperatura del servidor entre 18°C y 24°C, consumiendo menos energía que el sistema convencional.



Formulación del problema como Reinforcement Learning

Estado (State)



Cada estado debe contener al menos:



temperatura del servidor

temperatura exterior

número de usuarios

tasa de transferencia de datos



Todas las variables deben normalizarse antes de ingresar a la red neuronal.



Acciones (Action Space)



Utilizar un espacio de acciones discreto.



Ejemplo de cinco acciones:



enfriar mucho

enfriar poco

mantener temperatura

calentar poco

calentar mucho



Cada acción modifica la temperatura objetivo del sistema.



Recompensa (Reward)



La recompensa debe incentivar:



menor consumo energético

mantener la temperatura dentro del rango seguro

evitar sobrecalentamiento

evitar enfriamiento excesivo



Por ejemplo:



recompensa positiva cuando se ahorra energía

fuerte penalización cuando la temperatura supera 24°C

fuerte penalización cuando baja de 18°C

Algoritmo



Implementar Proximal Policy Optimization (PPO).



El entrenamiento debe seguir el algoritmo PPO estándar.



Implementar:



Actor

Critic

Generalized Advantage Estimation (GAE)

Clipped Objective

Entropy Bonus

Discount Factor (γ)

Lambda para GAE

Multiple Epoch Updates

Mini-batches

Arquitectura de la red neuronal



Implementar dos redes independientes:



Actor



Entrada:



vector de estado normalizado



Capas:



Dense 128

ReLU

Dense 64

ReLU

Dense 32

ReLU



Salida:



Distribución de probabilidad sobre las acciones mediante Softmax.



Critic



Entrada:



vector de estado normalizado



Capas:



Dense 128

ReLU

Dense 64

ReLU

Dense 32

ReLU



Salida:



Valor V(s)



Entorno



Implementar un entorno tipo Gymnasium.



Debe contener:



reset()



step(action)



render()



close()



El entorno debe simular:



temperatura exterior

usuarios

tráfico de datos

temperatura del servidor

consumo energético

consumo del sistema tradicional

consumo del sistema PPO

Simulación



Cada episodio representa un año completo.



Un paso representa un minuto.



Total:



365 días × 24 horas × 60 minutos



El entrenamiento debe permitir ejecutar múltiples episodios.



Métricas



Registrar durante el entrenamiento:



recompensa acumulada

consumo energético PPO

consumo energético tradicional

porcentaje de ahorro

temperatura media

temperatura máxima

temperatura mínima

pérdidas del Actor

pérdidas del Critic

entropía de la política

Visualizaciones



Generar gráficos de:



recompensa por episodio

ahorro energético

temperatura del servidor

consumo energético

pérdida del Actor

pérdida del Critic

evolución de la política

Comparación final



Comparar:



Sistema tradicional vs PPO



Mostrar:



consumo total

porcentaje de ahorro

temperatura media

desviación estándar de temperatura

número de violaciones del rango seguro

estabilidad del sistema

Tecnologías



Utilizar exclusivamente:



Python

PyTorch

Gymnasium

NumPy

Matplotlib

Pandas



No utilizar TensorFlow. 



In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class DataCenterEnv(gym.Env):
    """
    Entorno de Gymnasium para la simulación térmica y energética de un Data Center.
    Un paso (step) equivale a 1 minuto. Un episodio completo es 1 año (525,600 pasos).
    """
    metadata = {"render_modes": ["human"]}

    def __init__(self, max_steps=525600):
        super(DataCenterEnv, self).__init__()
        
        self.max_steps = max_steps
        
        # Espacio de acciones discreto (5 acciones)
        # 0: Enfriar mucho (-2°C), 1: Enfriar poco (-1°C), 2: Mantener (0°C), 3: Calentar poco (+1°C), 4: Calentar mucho (+2°C)
        self.action_space = spaces.Discrete(5)
        self.action_mapping = {0: -2.0, 1: -1.0, 2: 0.0, 3: 1.0, 4: 2.0}
        
        # Espacio de observaciones (Estado):
        # [Temp Servidor, Temp Exterior, Usuarios, Tráfico Datos]
        # Límites aproximados para normalización interna si fuera necesario
        low_obs = np.array([0.0, -10.0, 0.0, 0.0], dtype=np.float32)
        high_obs = np.array([80.0, 50.0, 1000.0, 100.0], dtype=np.float32)
        self.observation_space = spaces.Box(low=low_obs, high=high_obs, dtype=np.float32)
        
        # Rango seguro de operación
        self.T_MIN = 18.0
        self.T_MAX = 24.0
        
        self.reset()

    def _get_ambient_temp(self, step):
        # Simulación burda de ciclo diario y estacional (minutos en un año)
        minutes_in_day = 24 * 60
        minutes_in_year = self.max_steps
        
        # Ciclo anual (Estaciones)
        annual_cycle = np.sin(2 * np.pi * step / minutes_in_year) * 15.0 + 15.0 # Rango [0, 30]
        # Ciclo diario (Día/Noche)
        daily_cycle = np.sin(2 * np.pi * step / minutes_in_day) * 5.0
        
        return annual_cycle + daily_cycle

    def _get_server_load(self):
        # Usuarios simulados (0 a 1000) y tráfico de datos (0 a 100 Gbps)
        users = np.random.randint(100, 1000)
        traffic = (users / 1000.0) * 80.0 + np.random.uniform(0, 20)
        return float(users), float(traffic)

    def _simulate_server_temp(self, current_temp, ambient_temp, users, traffic):
        # Combinación lineal que dicta la inercia térmica natural del servidor
        heat_generated = 0.005 * users + 0.04 * traffic
        dissipation = 0.01 * (ambient_temp - current_temp)
        new_temp = current_temp + heat_generated + dissipation
        return float(new_temp)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        
        # Estado inicial
        self.ambient_temp = self._get_ambient_temp(self.current_step)
        self.users, self.traffic = self._get_server_load()
        self.server_temp = 21.0 # Empezar en el centro del rango óptimo
        
        # Métricas de consumo acumuladas
        self.total_energy_ppo = 0.0
        self.total_energy_trad = 0.0
        self.range_violations = 0
        
        state = np.array([self.server_temp, self.ambient_temp, self.users, self.traffic], dtype=np.float32)
        return self._normalize_state(state), {}

    def _normalize_state(self, state):
        # Normalización Z-score aproximada o MinMax basada en máximos teóricos
        norm_state = np.zeros_like(state)
        norm_state[0] = (state[0] - 21.0) / 5.0     # Temp Servidor
        norm_state[1] = (state[1] - 15.0) / 15.0    # Temp Exterior
        norm_state[2] = (state[2] - 500.0) / 300.0  # Usuarios
        norm_state[3] = (state[3] - 50.0) / 30.0    # Tráfico
        return norm_state

    def step(self, action):
        # 1. Capturar estado previo de temperatura del servidor
        prev_server_temp = self.server_temp
        
        # 2. Dinámica del entorno natural
        self.ambient_temp = self._get_ambient_temp(self.current_step)
        self.users, self.traffic = self._get_server_load()
        natural_temp = self._simulate_server_temp(prev_server_temp, self.ambient_temp, self.users, self.traffic)
        
        # 3. Aplicar Acción del Agente (Modificación térmica artificial)
        temp_change_agent = self.action_mapping[int(action)]
        self.server_temp = natural_temp + temp_change_agent
        
        # 4. Simulación del Sistema Tradicional (reacciona para emular mantener a 21°C constantes)
        trad_target = 21.0
        trad_action_needed = trad_target - natural_temp
        # El sistema tradicional tiene un límite físico de corrección por minuto similar al agente
        trad_action_applied = np.clip(trad_action_needed, -2.0, 2.0)
        trad_server_temp = natural_temp + trad_action_applied
        
        # 5. Cálculo de Consumo Energético (Proporcional al esfuerzo absoluto realizado)
        energy_ppo = abs(temp_change_agent) * 0.5  # Costo base por alteración de temperatura
        energy_trad = abs(trad_action_applied) * 0.5
        
        # Penalización si el agente se ve obligado a usar energía extrema de manera ineficiente
        # Si la temperatura natural ya estaba en rango, cambios drásticos se penalizan levemente en consumo.
        self.total_energy_ppo += energy_ppo
        self.total_energy_trad += energy_trad
        
        # 6. Cálculo de Recompensa (Reward)
        reward = 0.0
        
        # Incentivo de ahorro de energía frente al tradicional
        reward += (energy_trad - energy_ppo) * 2.0
        
        # Penalizaciones por violaciones de rango del servidor PPO
        if self.server_temp > self.T_MAX:
            violation_delta = self.server_temp - self.T_MAX
            reward -= (20.0 + 10.0 * violation_delta)
            self.range_violations += 1
        elif self.server_temp < self.T_MIN:
            violation_delta = self.T_MIN - self.server_temp
            reward -= (20.0 + 10.0 * violation_delta)
            self.range_violations += 1
        else:
            # Recompensa por mantener el rango seguro de manera estable
            reward += 5.0
            
        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        truncated = False
        
        next_state = np.array([self.server_temp, self.ambient_temp, self.users, self.traffic], dtype=np.float32)
        info = {
            "raw_server_temp": self.server_temp,
            "energy_ppo": energy_ppo,
            "energy_trad": energy_trad,
            "violation": (self.server_temp > self.T_MAX or self.server_temp < self.T_MIN)
        }
        
        return self._normalize_state(next_state), reward, terminated, truncated, info

    def render(self):
        print(f"Paso: {self.current_step} | Temp Servidor: {self.server_temp:.2°C} | Consumo Acumulado PPO: {self.total_energy_ppo:.2f}")

    def close(self):
        pass

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np


class ActorNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, action_dim),
            nn.Softmax(dim=-1),
        )

    def forward(self, state):
        return self.network(state)


class CriticNetwork(nn.Module):
    def __init__(self, state_dim):
        super(CriticNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, state):
        return self.network(state)


class PPOBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.is_terminals = []
        self.values = []

    def clear(self):
        del self.states[:]
        del self.actions[:]
        del self.log_probs[:]
        del self.rewards[:]
        del self.is_terminals[:]
        del self.values[:]


class PPOAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        lr_actor=3e-4,
        lr_critic=1e-3,
        gamma=0.99,
        K_epochs=4,
        eps_clip=0.2,
        lmbda=0.95,
        ent_coef=0.01,
    ):
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.K_epochs = K_epochs
        self.lmbda = lmbda
        self.ent_coef = ent_coef

        self.buffer = PPOBuffer()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.actor = ActorNetwork(state_dim, action_dim).to(self.device)
        self.critic = CriticNetwork(state_dim).to(self.device)

        self.optimizer_actor = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.optimizer_critic = optim.Adam(self.critic.parameters(), lr=lr_critic)

        self.MseLoss = nn.MSELoss()

    def select_action(self, state):
        state = torch.FloatTensor(state).to(self.device)
        with torch.no_grad():
            probs = self.actor(state)
            value = self.critic(state)

        dist = Categorical(probs)
        action = dist.sample()
        action_log_prob = dist.log_prob(action)

        self.buffer.states.append(state.cpu().numpy())
        self.buffer.actions.append(action.item())
        self.buffer.log_probs.append(action_log_prob.item())
        self.buffer.values.append(value.item())

        return action.item()

    def update(self):
        # Convertir listas del buffer a tensores de PyTorch
        old_states = torch.FloatTensor(np.array(self.buffer.states)).to(self.device)
        old_actions = torch.LongTensor(np.array(self.buffer.actions)).to(self.device)
        old_log_probs = torch.FloatTensor(np.array(self.buffer.log_probs)).to(
            self.device
        )
        old_values = np.array(self.buffer.values)
        rewards = self.buffer.rewards
        is_terminals = self.buffer.is_terminals

        # 1. Calcular Retornos y Ventajas usando GAE (Generalized Advantage Estimation)
        returns = []
        gae = 0
        # Añadimos un valor 0 simulado para el fin de la trayectoria
        next_value = 0

        # Recorrido inverso para GAE
        advantages = np.zeros_like(rewards, dtype=np.float32)
        for t in reversed(range(len(rewards))):
            if t == len(rewards) - 1:
                next_non_terminal = 1.0 - is_terminals[t]
                next_values = next_value
            else:
                next_non_terminal = 1.0 - is_terminals[t]
                next_values = old_values[t + 1]

            delta = (
                rewards[t]
                + self.gamma * next_values * next_non_terminal
                - old_values[t]
            )
            gae = delta + self.gamma * self.lmbda * next_non_terminal * gae
            advantages[t] = gae

        returns = advantages + old_values

        # Convertir a tensores
        advantages = torch.FloatTensor(advantages).to(self.device)
        returns = torch.FloatTensor(returns).to(self.device)

        # Normalizar ventajas para dar estabilidad al gradiente
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        actor_losses, critic_losses, entropy_losses = [], [], []

        # 2. Optimización por épocas (Multiple Epoch Updates con Mini-batches implícitos)
        for _ in range(self.K_epochs):
            # Evaluar las acciones viejas bajo la política actual
            probs = self.actor(old_states)
            dist = Categorical(probs)
            log_probs = dist.log_prob(old_actions)
            entropy = dist.entropy()
            state_values = self.critic(old_states).squeeze()

            # Ratio de probabilidades r_t(θ)
            ratios = torch.exp(log_probs - old_log_probs)

            # Clipped Surrogate Loss (PPO Objective)
            surr1 = ratios * advantages
            surr2 = (
                torch.clamp(ratios, 1.0 - self.eps_clip, 1.0 + self.eps_clip)
                * advantages
            )

            actor_loss = (
                -torch.min(surr1, surr2).mean() - self.ent_coef * entropy.mean()
            )
            critic_loss = self.MseLoss(state_values, returns)

            # Actualización de gradientes del Actor
            self.optimizer_actor.zero_grad()
            actor_loss.backward()
            self.optimizer_actor.step()

            # Actualización de gradientes del Critic
            self.optimizer_critic.zero_grad()
            critic_loss.backward()
            self.optimizer_critic.step()

            actor_losses.append(actor_loss.item())
            critic_losses.append(critic_loss.item())
            entropy_losses.append(entropy.mean().item())

        self.buffer.clear()

        return np.mean(actor_losses), np.mean(critic_losses), np.mean(entropy_losses)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def run_experiment(episodes=5, steps_per_episode=10000):
    """
    Ejecuta el entrenamiento del Agente PPO en el entorno del Data Center.
    Reducimos el tamaño del episodio de simulación por motivos prácticos de demostración de convergencia.
    """
    env = DataCenterEnv(max_steps=steps_per_episode)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = PPOAgent(
        state_dim=state_dim, action_dim=action_dim, K_epochs=5, ent_coef=0.02
    )

    # DataFrames / Diccionarios para métricas historiadas
    history = {
        "episode": [],
        "reward": [],
        "energy_ppo": [],
        "energy_trad": [],
        "saving_pct": [],
        "temp_mean": [],
        "temp_max": [],
        "temp_min": [],
        "actor_loss": [],
        "critic_loss": [],
        "entropy": [],
        "violations": [],
    }

    print("--- Iniciando Entrenamiento PPO del Centro de Datos ---")

    for ep in range(1, episodes + 1):
        state, _ = env.reset()
        ep_reward = 0
        temps = []

        # Almacenamiento temporal para actualizar al agente al final del episodio o en ventanas fixed
        update_timestep = 0
        update_every = 2000  # Frecuencia de actualización de la red

        for step in range(steps_per_episode):
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, info = env.step(action)

            agent.buffer.rewards.append(reward)
            agent.buffer.is_terminals.append(terminated)

            state = next_state
            ep_reward += reward
            temps.append(info["raw_server_temp"])

            update_timestep += 1
            if update_timestep % update_every == 0 or terminated:
                a_loss, c_loss, ent = agent.update()

            if terminated:
                break

        # Procesar métricas del episodio completo
        pct_saving = (
            (env.total_energy_trad - env.total_energy_ppo)
            / (env.total_energy_trad + 1e-8)
        ) * 100

        history["episode"].append(ep)
        history["reward"].append(ep_reward)
        history["energy_ppo"].append(env.total_energy_ppo)
        history["energy_trad"].append(env.total_energy_trad)
        history["saving_pct"].append(pct_saving)
        history["temp_mean"].append(np.mean(temps))
        history["temp_max"].append(np.max(temps))
        history["temp_min"].append(np.min(temps))
        history["actor_loss"].append(a_loss)
        history["critic_loss"].append(c_loss)
        history["entropy"].append(ent)
        history["violations"].append(env.range_violations)

        print(
            f"Episodio {ep}/{episodes} | Recompensa: {ep_reward:.1f} | Ahorro Energético: {pct_saving:.2f}% | Violaciones Rango: {env.range_violations}"
        )

    return pd.DataFrame(history), env, temps


def plot_results(df, final_temps):
    # Gráficos Solicitados
    plt.figure(figsize=(16, 12))

    # 1. Recompensa por Episodio
    plt.subplot(3, 2, 1)
    plt.plot(df["episode"], df["reward"], marker="o", color="purple")
    plt.title("Recompensa Acumulada por Episodio")
    plt.xlabel("Episodio")
    plt.ylabel("Recompensa")
    plt.grid(True)

    # 2. Ahorro Energético Porcentual
    plt.subplot(3, 2, 2)
    plt.plot(df["episode"], df["saving_pct"], marker="s", color="green")
    plt.title("Evolución del Ahorro Energético vs Sistema Tradicional")
    plt.xlabel("Episodio")
    plt.ylabel("% de Ahorro")
    plt.grid(True)

    # 3. Evolución de Pérdidas de las Redes
    plt.subplot(3, 2, 3)
    plt.plot(df["episode"], df["actor_loss"], label="Actor Loss", color="blue")
    plt.plot(df["episode"], df["critic_loss"], label="Critic Loss", color="orange")
    plt.title("Evolución de Pérdidas de Redes Independientes")
    plt.xlabel("Episodio")
    plt.ylabel("Pérdida")
    plt.legend()
    plt.grid(True)

    # 4. Entropía de la Política
    plt.subplot(3, 2, 4)
    plt.plot(df["episode"], df["entropy"], color="magenta")
    plt.title("Evolución de la Entropía de la Política")
    plt.xlabel("Episodio")
    plt.ylabel("Entropía")
    plt.grid(True)

    # 5. Comportamiento Térmico del último Episodio entrenado
    plt.subplot(3, 2, (5, 6))
    plt.plot(
        final_temps[:1440], label="Temperatura Servidor (PPO)", color="crimson"
    )  # Muestra de las primeras 24 horas (1440 mins)
    plt.axhline(24, color="black", linestyle="--", label="Límite Máximo Seguro (24°C)")
    plt.axhline(18, color="black", linestyle="--", label="Límite Mínimo Seguro (18°C)")
    plt.title(
        "Perfil de Temperatura Interna del Servidor en Último Episodio (Primeras 24 Horas)"
    )
    plt.xlabel("Minutos")
    plt.ylabel("Temperatura (°C)")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig("metricas_ppo_datacenter.png")
    plt.show()


if __name__ == "__main__":
    # Ejecutamos una simulación representativa (e.g. 5 episodios completos de 15,000 pasos c/u para balance velocidad/precisión)
    df_metrics, final_env, last_episode_temps = run_experiment(
        episodes=6, steps_per_episode=20000
    )

    # Generar Visualizaciones
    plot_results(df_metrics, last_episode_temps)

    # Tabla Comparativa Final
    print("\n" + "=" * 50)
    print("      COMPARATIVA FINAL: SISTEMA TRADICIONAL VS PPO")
    print("=" * 50)
    last_row = df_metrics.iloc[-1]

    print(f"Consumo Total Sistema Tradicional : {last_row['energy_trad']:.2f} Unidades")
    print(f"Consumo Total Agente PPO          : {last_row['energy_ppo']:.2f} Unidades")
    print(f"Porcentaje de Ahorro Neto         : {last_row['saving_pct']:.2f} %")
    print(f"Temperatura Media del Servidor     : {last_row['temp_mean']:.2f} °C")
    print(
        f"Rango Registrado [Mín, Máx]       : [{last_row['temp_min']:.1f}°C, {last_row['temp_max']:.1f}°C]"
    )
    print(f"Número de Violaciones de Seguridad : {int(last_row['violations'])}")
    print(f"Estabilidad del Agente (Entropía)  : {last_row['entropy']:.4f}")
    print("=" * 50)

podrias realiizar el Agente PPO por algun framework conocido como stablebaselines

In [ ]:
!pip install stable-baselines3 gymnasium numpy pandas matplotlib

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback


# ==========================================
# 1. ENTORNO COMPATIBLE CON GYMNASIUM
# ==========================================
class DataCenterEnvSB3(gym.Env):
    """
    Entorno optimizado para acoplarse con Stable-Baselines3.
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, max_steps=20000):
        super(DataCenterEnvSB3, self).__init__()
        self.max_steps = max_steps

        # 5 Acciones discretas
        self.action_space = spaces.Discrete(5)
        self.action_mapping = {0: -2.0, 1: -1.0, 2: 0.0, 3: 1.0, 4: 2.0}

        # Espacio de observaciones continuo [Temp Servidor, Temp Exterior, Usuarios, Tráfico Datos]
        low_obs = np.array([-5.0, -5.0, -5.0, -5.0], dtype=np.float32)
        high_obs = np.array([5.0, 5.0, 5.0, 5.0], dtype=np.float32)
        self.observation_space = spaces.Box(
            low=low_obs, high=high_obs, dtype=np.float32
        )

        self.T_MIN = 18.0
        self.T_MAX = 24.0
        self.reset()

    def _get_ambient_temp(self, step):
        minutes_in_day = 24 * 60
        annual_cycle = np.sin(2 * np.pi * step / self.max_steps) * 15.0 + 15.0
        daily_cycle = np.sin(2 * np.pi * step / minutes_in_day) * 5.0
        return annual_cycle + daily_cycle

    def _get_server_load(self):
        users = np.random.randint(100, 1000)
        traffic = (users / 1000.0) * 80.0 + np.random.uniform(0, 20)
        return float(users), float(traffic)

    def _normalize_state(self, state):
        norm_state = np.zeros_like(state)
        norm_state[0] = (state[0] - 21.0) / 5.0
        norm_state[1] = (state[1] - 15.0) / 15.0
        norm_state[2] = (state[2] - 500.0) / 300.0
        norm_state[3] = (state[3] - 50.0) / 30.0
        return norm_state

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.ambient_temp = self._get_ambient_temp(self.current_step)
        self.users, self.traffic = self._get_server_load()
        self.server_temp = 21.0

        self.total_energy_ppo = 0.0
        self.total_energy_trad = 0.0
        self.range_violations = 0

        state = np.array(
            [self.server_temp, self.ambient_temp, self.users, self.traffic],
            dtype=np.float32,
        )
        return self._normalize_state(state), {}

    def step(self, action):
        prev_server_temp = self.server_temp
        self.ambient_temp = self._get_ambient_temp(self.current_step)
        self.users, self.traffic = self._get_server_load()

        # Dinámica natural
        heat_generated = 0.005 * self.users + 0.04 * self.traffic
        dissipation = 0.01 * (self.ambient_temp - prev_server_temp)
        natural_temp = prev_server_temp + heat_generated + dissipation

        # Acción del Agente PPO
        temp_change_agent = self.action_mapping[int(action)]
        self.server_temp = natural_temp + temp_change_agent

        # Sistema Tradicional (Mantiene a 21°C constantes dentro de los límites del compresor)
        trad_action_applied = np.clip(21.0 - natural_temp, -2.0, 2.0)

        # Energía consumida (Esfuerzo mecánico)
        energy_ppo = abs(temp_change_agent) * 0.5
        energy_trad = abs(trad_action_applied) * 0.5

        self.total_energy_ppo += energy_ppo
        self.total_energy_trad += energy_trad

        # Función de Recompensa
        reward = 0.0
        reward += (energy_trad - energy_ppo) * 2.0  # Incentivo de ahorro energético

        if self.server_temp > self.T_MAX:
            reward -= 20.0 + 10.0 * (self.server_temp - self.T_MAX)
            self.range_violations += 1
        elif self.server_temp < self.T_MIN:
            reward -= 20.0 + 10.0 * (self.T_MIN - self.server_temp)
            self.range_violations += 1
        else:
            reward += 5.0  # Estabilidad en zona de confort

        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        truncated = False

        next_state = np.array(
            [self.server_temp, self.ambient_temp, self.users, self.traffic],
            dtype=np.float32,
        )

        info = {
            "server_temp": self.server_temp,
            "energy_ppo": energy_ppo,
            "energy_trad": energy_trad,
            "violations": self.range_violations,
        }

        return self._normalize_state(next_state), reward, terminated, truncated, info


# ==========================================
# 2. CALLBACK PARA MONITOREAR MÉTRICAS
# ==========================================
class DataCenterMetricsCallback(BaseCallback):
    """Callback personalizado para capturar datos de rendimiento en SB3"""

    def __init__(self, verbose=0):
        super(DataCenterMetricsCallback, self).__init__(verbose)
        self.episodes_data = []

    def _on_step(self) -> bool:
        for info in self.locals["infos"]:
            if "episode" in info.keys():
                # CAMBIO AQUÍ: Agregamos .unwrapped al final para acceder al entorno real
                env = self.training_env.envs[0].unwrapped

                pct_saving = (
                    (env.total_energy_trad - env.total_energy_ppo)
                    / (env.total_energy_trad + 1e-8)
                ) * 100

                self.episodes_data.append(
                    {
                        "reward": info["episode"]["r"],
                        "energy_ppo": env.total_energy_ppo,
                        "energy_trad": env.total_energy_trad,
                        "saving_pct": pct_saving,
                        "violations": env.range_violations,
                    }
                )
        return True


# ==========================================
# 3. PIPELINE DE ENTRENAMIENTO Y COMPARATIVA
# ==========================================
if __name__ == "__main__":
    steps_per_episode = 20000
    total_episodes = 6
    total_timesteps = steps_per_episode * total_episodes

    # Crear Entorno
    env = DataCenterEnvSB3(max_steps=steps_per_episode)

    # Configurar Arquitectura de Red (MlpPolicy: Redes Independientes Actor/Critic)
    # net_arch=dict(pi=[128, 64, 32], vf=[128, 64, 32]) define las capas Dense solicitadas
    policy_kwargs = dict(net_arch=dict(pi=[128, 64, 32], vf=[128, 64, 32]))

    # Instanciar PPO con parámetros GAE, Clipped Objective y Entropía
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=3e-4,
        n_steps=2000,  # Frecuencia de actualización (Mini-batches)
        batch_size=64,
        n_epochs=5,  # Multiple Epoch Updates
        gamma=0.99,  # Discount Factor
        gae_lambda=0.95,  # Lambda para GAE
        clip_range=0.2,  # Clipped Objective
        ent_coef=0.02,  # Entropy Bonus
        policy_kwargs=policy_kwargs,
        verbose=1,
    )

    # Entrenar el Modelo
    callback = DataCenterMetricsCallback()
    print("--- Iniciando Entrenamiento con Stable-Baselines3 ---")
    model.learn(total_timesteps=total_timesteps, callback=callback)

    # Procesar histórico de resultados
    df_metrics = pd.DataFrame(callback.episodes_data)

    # ==========================================
    # 4. EVALUACIÓN Y RECOLECCIÓN DE HISTORIAL TÉRMICO
    # ==========================================
    obs, _ = env.reset()
    final_temps = []

    for _ in range(steps_per_episode):
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        final_temps.append(info["server_temp"])
        if terminated:
            break

    # ==========================================
    # 5. VISUALIZACIONES
    # ==========================================
    plt.figure(figsize=(14, 10))

    # Recompensas
    plt.subplot(2, 2, 1)
    plt.plot(df_metrics.index + 1, df_metrics["reward"], marker="o", color="teal")
    plt.title("Recompensa por Episodio (SB3)")
    plt.xlabel("Episodio")
    plt.grid(True)

    # Ahorro energético
    plt.subplot(2, 2, 2)
    plt.plot(
        df_metrics.index + 1, df_metrics["saving_pct"], marker="s", color="forestgreen"
    )
    plt.title("Evolución de Ahorro Energético (%)")
    plt.xlabel("Episodio")
    plt.grid(True)

    # Comportamiento Térmico del último Episodio
    plt.subplot(2, 2, (3, 4))
    plt.plot(
        final_temps[:1440], label="Temperatura Servidor PPO (SB3)", color="crimson"
    )
    plt.axhline(24, color="black", linestyle="--", label="Límite Máximo (24°C)")
    plt.axhline(18, color="black", linestyle="--", label="Límite Mínimo (18°C)")
    plt.title(
        "Perfil Térmico - Primeras 24 Horas de Simulación Posterior al Entrenamiento"
    )
    plt.xlabel("Minutos")
    plt.ylabel("Temperatura (°C)")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig("metricas_sb3_datacenter.png")
    plt.show()

    # ==========================================
    # 6. COMPARATIVA FINAL IMPRESA
    # ==========================================
    print("\n" + "=" * 50)
    print("      COMPARATIVA FINAL (STABLE-BASELINES3 VS TRADICIONAL)")
    print("=" * 50)
    if not df_metrics.empty:
        last_ep = df_metrics.iloc[-1]
        print(
            f"Consumo Total Tradicional         : {last_ep['energy_trad']:.2f} Unidades"
        )
        print(
            f"Consumo Total Agente PPO (SB3)    : {last_ep['energy_ppo']:.2f} Unidades"
        )
        print(f"Porcentaje de Ahorro Neto         : {last_ep['saving_pct']:.2f} %")
        print(f"Número de Violaciones de Seguridad : {int(last_ep['violations'])}")
    print("=" * 50)

podrias utlizar gymnasium, SB2 y un espacio de acciones continuo para este problema?

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback


# ==========================================
# 1. ENTORNO CONTINUO CON GYMNASIUM
# ==========================================
class DataCenterContinuousEnv(gym.Env):
    """
    Entorno con Espacio de Acciones Continuo.
    La acción representa directamente el cambio térmico forzado por minuto.
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, max_steps=20000):
        super(DataCenterContinuousEnv, self).__init__()
        self.max_steps = max_steps

        # ESPACIO DE ACCIONES CONTINUO: Rango [-2.0, 2.0]
        # Representa los grados centígrados (°C) a modificar en un minuto.
        self.action_space = spaces.Box(low=-2.0, high=2.0, shape=(1,), dtype=np.float32)

        # Espacio de observaciones continuo normatizado
        self.observation_space = spaces.Box(
            low=np.array([-5.0, -5.0, -5.0, -5.0], dtype=np.float32),
            high=np.array([5.0, 5.0, 5.0, 5.0], dtype=np.float32),
            dtype=np.float32,
        )

        self.T_MIN = 18.0
        self.T_MAX = 24.0
        self.reset()

    def _get_ambient_temp(self, step):
        minutes_in_day = 24 * 60
        annual_cycle = np.sin(2 * np.pi * step / self.max_steps) * 15.0 + 15.0
        daily_cycle = np.sin(2 * np.pi * step / minutes_in_day) * 5.0
        return annual_cycle + daily_cycle

    def _get_server_load(self):
        users = np.random.randint(100, 1000)
        traffic = (users / 1000.0) * 80.0 + np.random.uniform(0, 20)
        return float(users), float(traffic)

    def _normalize_state(self, state):
        norm_state = np.zeros_like(state)
        norm_state[0] = (state[0] - 21.0) / 5.0
        norm_state[1] = (state[1] - 15.0) / 15.0
        norm_state[2] = (state[2] - 500.0) / 300.0
        norm_state[3] = (state[3] - 50.0) / 30.0
        return norm_state

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.ambient_temp = self._get_ambient_temp(self.current_step)
        self.users, self.traffic = self._get_server_load()
        self.server_temp = 21.0

        self.total_energy_ppo = 0.0
        self.total_energy_trad = 0.0
        self.range_violations = 0

        state = np.array(
            [self.server_temp, self.ambient_temp, self.users, self.traffic],
            dtype=np.float32,
        )
        return self._normalize_state(state), {}

    def step(self, action):
        prev_server_temp = self.server_temp
        self.ambient_temp = self._get_ambient_temp(self.current_step)
        self.users, self.traffic = self._get_server_load()

        # Dinámica térmica natural del servidor
        heat_generated = 0.005 * self.users + 0.04 * self.traffic
        dissipation = 0.01 * (self.ambient_temp - prev_server_temp)
        natural_temp = prev_server_temp + heat_generated + dissipation

        # Extraer el valor escalar continuo de la acción (vienen dentro de un array de NumPy)
        temp_change_agent = float(action[0])
        # Asegurar físicamente los límites de la acción por si el agente se desvía en la exploración
        temp_change_agent = np.clip(temp_change_agent, -2.0, 2.0)

        # Aplicar la acción
        self.server_temp = natural_temp + temp_change_agent

        # Sistema Tradicional Proporcional (Intenta devolver el sistema a 21.0°C de forma exacta)
        trad_action_needed = 21.0 - natural_temp
        trad_action_applied = np.clip(trad_action_needed, -2.0, 2.0)

        # Cálculo de consumo proporcional al esfuerzo absoluto continuo
        energy_ppo = abs(temp_change_agent) * 0.5
        energy_trad = abs(trad_action_applied) * 0.5

        self.total_energy_ppo += energy_ppo
        self.total_energy_trad += energy_trad

        # Cálculo de Recompensas
        reward = 0.0
        reward += (energy_trad - energy_ppo) * 2.5  # Peso al ahorro energético

        # Fuertes penalizaciones por violaciones de rango seguro
        if self.server_temp > self.T_MAX:
            reward -= 25.0 + 15.0 * (self.server_temp - self.T_MAX)
            self.range_violations += 1
        elif self.server_temp < self.T_MIN:
            reward -= 25.0 + 15.0 * (self.T_MIN - self.server_temp)
            self.range_violations += 1
        else:
            # Recompensa por mantener la estabilidad en la zona de confort
            reward += 5.0

        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        truncated = False

        next_state = np.array(
            [self.server_temp, self.ambient_temp, self.users, self.traffic],
            dtype=np.float32,
        )

        info = {
            "server_temp": self.server_temp,
            "energy_ppo": energy_ppo,
            "energy_trad": energy_trad,
            "violations": self.range_violations,
        }

        return self._normalize_state(next_state), reward, terminated, truncated, info


# ==========================================
# 2. CALLBACK DE MONITOREO
# ==========================================
class ContinuousMetricsCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(ContinuousMetricsCallback, self).__init__(verbose)
        self.episodes_data = []

    def _on_step(self) -> bool:
        for info in self.locals["infos"]:
            if "episode" in info.keys():
                env = self.training_env.envs[0]
                pct_saving = (
                    (env.total_energy_trad - env.total_energy_ppo)
                    / (env.total_energy_trad + 1e-8)
                ) * 100

                self.episodes_data.append(
                    {
                        "reward": info["episode"]["r"],
                        "energy_ppo": env.total_energy_ppo,
                        "energy_trad": env.total_energy_trad,
                        "saving_pct": pct_saving,
                        "violations": env.range_violations,
                    }
                )
        return True


# ==========================================
# 3. PIPELINE PRINCIPAL DE EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    steps_per_episode = 20000
    total_episodes = 6
    total_timesteps = steps_per_episode * total_episodes

    env = DataCenterContinuousEnv(max_steps=steps_per_episode)

    # Arquitectura de capas independientes: Dense 128 -> 64 -> 32
    policy_kwargs = dict(net_arch=dict(pi=[128, 64, 32], vf=[128, 64, 32]))

    # Configuración de PPO para espacios continuos
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=3e-4,
        n_steps=2000,
        batch_size=64,
        n_epochs=5,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,  # Un coeficiente menor suele ayudar en continuo para que la varianza no explote
        policy_kwargs=policy_kwargs,
        verbose=1,
    )

    print("--- Entrenando PPO Continuo con Stable-Baselines3 ---")
    callback = ContinuousMetricsCallback()
    model.learn(total_timesteps=total_timesteps, callback=callback)

    df_metrics = pd.DataFrame(callback.episodes_data)

    # Evaluación post-entrenamiento (Guardando trayectoria térmica)
    obs, _ = env.reset()
    final_temps = []
    for _ in range(steps_per_episode):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        final_temps.append(info["server_temp"])
        if terminated:
            break

    # ==========================================
    # 4. GRÁFICOS Y RESULTADOS
    # ==========================================
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(
        df_metrics.index + 1, df_metrics["saving_pct"], marker="o", color="dodgerblue"
    )
    plt.title("Porcentaje de Ahorro Energético Continuo")
    plt.xlabel("Episodio")
    plt.ylabel("% de Ahorro")
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(final_temps[:1440], color="orangered", label="Temperatura Servidor PPO")
    plt.axhline(24, color="black", linestyle="--", label="Límite Máximo (24°C)")
    plt.axhline(18, color="black", linestyle="--", label="Límite Mínimo (18°C)")
    plt.title("Ajuste Térmico Continuo Fino (Primeras 24 Horas)")
    plt.xlabel("Minutos")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig("metricas_ppo_continuo.png")
    plt.show()

    print("\n" + "=" * 50)
    print("      COMPARATIVA FINAL: AGENTE CONTINUO VS TRADICIONAL")
    print("=" * 50)
    if not df_metrics.empty:
        last_ep = df_metrics.iloc[-1]
        print(
            f"Consumo Total Tradicional         : {last_ep['energy_trad']:.2f} Unidades"
        )
        print(
            f"Consumo Total Agente PPO Continuo : {last_ep['energy_ppo']:.2f} Unidades"
        )
        print(f"Porcentaje de Ahorro Neto         : {last_ep['saving_pct']:.2f} %")
        print(f"Número de Violaciones de Seguridad : {int(last_ep['violations'])}")
    print("=" * 50)

necesitamos trimmear la distribucion por tangente hiperbolica para no exceder pasos fisicos de calentamiento o enfriamiento?

No es estrictamente obligatorio, pero sí es una excelente práctica de diseño en Reinforcement Learning Continuo para estabilizar el entrenamiento.Cuando trabajamos con un espacio de acciones continuo usando PPO, el agente modela la política como una distribución Gaussiana (Normal). Si dejamos esta distribución libre, teóricamente el agente podría muestrear valores extremos como $+5.0^\circ\text{C}$ o $-12.0^\circ\text{C}$ durante la fase de exploración (especialmente al inicio, cuando la desviación estándar $\sigma$ es alta), lo que rompería las restricciones físicas del compresor del Data Center.Aquí te explico cómo afecta esto y cuál es la forma correcta de implementarlo:1. El problema del truncamiento simple (np.clip)Si dejas que la red neuronal escupa cualquier valor, se genere la distribución Gaussiana, y luego simplemente aplicas un np.clip(accion, -2.0, 2.0) en el entorno:El problema del gradiente muerto: El agente puede proponer una acción de $+4.0$, pero el entorno la corta en $+2.0$. Para el entorno la acción fue $+2.0$, pero matemáticamente la política cree que su acción extrema fue la que obtuvo la recompensa. Esto descalibra el cálculo de la probabilidad $f(a|\mu, \sigma)$ y ralentiza severamente la convergencia (problema conocido como boundary effects).2. La solución óptima: Redirección por Tangente Hiperbólica ($\tanh$)La solución estándar en la industria (utilizada internamente por frameworks como Stable-Baselines3 o cuando se programa el algoritmo desde cero) consiste en aplicar una capa $\tanh$ directamente a la salida de la media de la red del Actor antes de construir la distribución, o bien transformar la muestra.Dado que la función $\tanh$ comprime cualquier valor real al rango exacto de $[-1.0, 1.0]$, solo dependemos de un factor de escala:$$\text{Acción física} = \tanh(\text{salida de la red}) \times \text{Max Step}$$En nuestro caso, como el límite físico es $2.0^\circ\text{C}$, multiplicamos por 2.0.3. Cómo se implementa (En código nativo de PyTorch)Si estuvieras construyendo el Actor desde cero en PyTorch, en lugar de mapear la salida linealmente, forzarías la media ($\mu$) a través de una función $\tanh$:

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Normal


class ContinuousActor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ContinuousActor, self).__init__()
        self.fc_layers = nn.Sequential(
            nn.Linear(state_dim, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU()
        )
        self.mu_head = nn.Linear(64, action_dim)
        # Log desviación estándar (se inicializa típicamente en 0, lo que da std = 1)
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, state):
        x = self.fc_layers(state)

        # 1. Trimmear la media usando Tanh
        # mu queda estrictamente entre -1 y 1
        mu = torch.tanh(self.mu_head(x))

        # 2. Escalar al rango físico real (-2.0°C a 2.0°C)
        mu = mu * 2.0

        std = torch.exp(self.log_std)

        # 3. Crear la distribución Gaussiana acotada en su centro
        dist = Normal(mu, std)
        return dist

¿Qué pasa con Stable-Baselines3?
Si estás utilizando la solución del paso anterior con Stable-Baselines3, no necesitas programar esto manualmente. SB3 detecta que tu action_space es un spaces.Box(low=-2.0, high=2.0) y, de manera automática en su arquitectura MlpPolicy para PPO, aplica la reparameterización y el escalado utilizando funciones similares para garantizar que la probabilidad matemática de la acción coincida perfectamente con los límites físicos del entorno.